# 📄 Agent 1: Document Classification Agent

**Purpose:** Reads a document and classifies it into one of these types:
- 📄 Invoice
- 📋 Contract
- 📊 Report
- 📝 Email
- ❓ Unknown

**Tech:** Gemini 1.5 Flash via **Vertex AI**

In [1]:
# Install dependencies (run once)
# !pip install google-genai python-dotenv

In [5]:
from google import genai
from google.genai import types

# Initialize client using Vertex AI (uses gcloud credentials)
client = genai.Client(
    vertexai=True,
    project="pwc-agentic-demo",
    location="us-central1"
)

print("✅ Vertex AI client initialized!")
print("   Project: pwc-agentic-demo")
print("   Location: us-central1")

✅ Vertex AI client initialized!
   Project: pwc-agentic-demo
   Location: us-central1


## 🧪 Test 1: Classify a Text Document

In [6]:
def classify_document(text: str) -> dict:
    """
    Classifies a document using Gemini 1.5 Flash on Vertex AI.
    Returns: {type, confidence, reasoning}
    """
    import json
    
    prompt = f"""You are a document classification expert.
Classify the following document into exactly ONE category:
- Invoice
- Contract
- Report
- Email
- Unknown

Respond in this exact JSON format:
{{"type": "<category>", "confidence": <0.0-1.0>, "reasoning": "<brief explanation>"}}

Document:
---
{text}
---"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.1,
            response_mime_type="application/json"
        )
    )

    return json.loads(response.text)

In [7]:
# Sample Invoice
sample_invoice = """
INVOICE #INV-2024-0042
Date: March 15, 2024

Bill To:
PricewaterhouseCoopers
5th Floor, Block B
Cyber Towers, Hyderabad

Description                    Qty    Rate      Amount
AI Consulting Services          40    $150/hr    $6,000.00
Cloud Infrastructure Setup       1    $3,500.00   $3,500.00
                                
                              Subtotal:   $9,500.00
                              Tax (18%):  $1,710.00
                              Total:     $11,210.00

Payment Terms: Net 30 days
Bank: HDFC Bank, A/C: 50100012345678
"""

result = classify_document(sample_invoice)
print(f"📄 Type: {result['type']}")
print(f"📊 Confidence: {result['confidence']}")
print(f"💡 Reasoning: {result['reasoning']}")

📄 Type: Invoice
📊 Confidence: 0.99
💡 Reasoning: The document explicitly states 'INVOICE #INV-2024-0042' and includes typical invoice elements such as 'Bill To', 'Description', 'Qty', 'Rate', 'Amount', 'Subtotal', 'Tax', 'Total', and 'Payment Terms'.


In [8]:
# Sample Contract
sample_contract = """
MASTER SERVICES AGREEMENT

This Agreement is entered into as of January 1, 2024, between:
ABC Technology Pvt. Ltd. ("Provider")
and
XYZ Consulting ("Client")

1. SCOPE OF SERVICES
Provider shall deliver AI-powered document processing solutions as detailed in Exhibit A.

2. TERM
This Agreement shall remain in effect for 12 months from the Effective Date.

3. COMPENSATION
Client shall pay Provider $150,000 annually, payable in quarterly installments.

4. CONFIDENTIALITY
Both parties agree to maintain strict confidentiality of all proprietary information.

5. TERMINATION
Either party may terminate with 30 days written notice.
"""

result = classify_document(sample_contract)
print(f"📄 Type: {result['type']}")
print(f"📊 Confidence: {result['confidence']}")
print(f"💡 Reasoning: {result['reasoning']}")

📄 Type: Contract
📊 Confidence: 1.0
💡 Reasoning: The document is explicitly titled 'MASTER SERVICES AGREEMENT' and contains standard contractual elements such as parties involved, scope of services, term, compensation, confidentiality clauses, and termination conditions.


In [9]:
# Sample Report
sample_report = """
QUARTERLY PERFORMANCE REPORT - Q4 2023

Executive Summary:
Revenue increased by 23% YoY to $4.2M. Client acquisition rate improved
by 15% compared to Q3. Employee retention stands at 92%.

Key Metrics:
- Total Revenue: $4,200,000
- New Clients: 12
- Client Satisfaction Score: 4.6/5.0
- Project Delivery On-Time Rate: 94%

Challenges:
1. Cloud infrastructure costs exceeded budget by 8%
2. Two senior engineers resigned in November

Recommendations:
- Negotiate better cloud pricing with AWS
- Implement retention bonuses for key personnel
"""

result = classify_document(sample_report)
print(f"📄 Type: {result['type']}")
print(f"📊 Confidence: {result['confidence']}")
print(f"💡 Reasoning: {result['reasoning']}")

📄 Type: Report
📊 Confidence: 1.0
💡 Reasoning: The document is explicitly titled 'QUARTERLY PERFORMANCE REPORT' and contains sections like 'Executive Summary', 'Key Metrics', 'Challenges', and 'Recommendations', which are all characteristic elements of a business report.


## 📁 Test 2: Classify a PDF File

In [11]:
def classify_pdf(pdf_path: str) -> dict:
    """
    Classifies a PDF document by reading its content.
    Uses PyMuPDF to extract text, then sends to Gemini.
    """
    import fitz  # PyMuPDF
    
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    
    # Truncate if too long
    if len(text) > 50000:
        text = text[:50000] + "\n\n[Document truncated...]"
    
    return classify_document(text)

In [19]:
# Test with a PDF (uncomment and provide path)
result = classify_pdf(r"C:\Users\madha\Downloads\Event Timings.pdf")
print(f"📄 Type: {result['type']}")
print(f"📊 Confidence: {result['confidence']}")
print(f"💡 Reasoning: {result['reasoning']}")

📄 Type: Unknown
📊 Confidence: 0.9
💡 Reasoning: The document details an 'Event Flow' with specific timings, activities, and speakers, resembling an event schedule or agenda. This format does not align with the characteristics of an Invoice, Contract, Report, or Email.


## 🧪 Test 3: Classify an Image (Multimodal)

Gemini can read images directly — no separate OCR needed!

In [22]:
def classify_image(image_path: str) -> dict:
    """
    Classifies a document from an image using Gemini's multimodal capabilities.
    """
    import base64
    import json
    
    with open(image_path, "rb") as f:
        image_data = f.read()
    
    # Detect mime type
    ext = image_path.lower().split(".")[-1]
    mime_map = {"png": "image/png", "jpg": "image/jpeg", "jpeg": "image/jpeg", "webp": "image/webp"}
    mime_type = mime_map.get(ext, "image/png")
    
    prompt = """You are a document classification expert.
Look at this image of a document and classify it into exactly ONE category:
- Invoice
- Contract
- Report
- Email
- Unknown

Respond in this exact JSON format:
{{"type": "<category>", "confidence": <0.0-1.0>, "reasoning": "<brief explanation>"}}"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            types.Part.from_bytes(data=image_data, mime_type=mime_type),
            prompt
        ],
        config=types.GenerateContentConfig(
            temperature=0.1,
            response_mime_type="application/json"
        )
    )
    
    return json.loads(response.text)

In [23]:
# Test with an image (uncomment and provide path)
result = classify_image(r"C:\Users\madha\Downloads\canva-black-and-gray-minimal-freelancer-invoice-wPpAXSlmfF4.webp")
print(f"📄 Type: {result['type']}")
print(f"📊 Confidence: {result['confidence']}")
print(f"💡 Reasoning: {result['reasoning']}")

📄 Type: Invoice
📊 Confidence: 1.0
💡 Reasoning: The document prominently displays 'INVOICE' as its title and contains all the typical elements of an invoice, including an invoice number, date, 'billed to' and 'from' sections, an itemized list with quantity, price, and amount, a total amount, and payment method details.


## ✅ Classification Agent - Complete

**What we built:**
- Text classification with structured JSON output
- PDF classification with text extraction
- Image classification with Gemini's multimodal OCR

**Next:** Extraction Agent (Agent 2) — extracts key fields based on document type